# TDT PLN — Prueba de Concepto
**MULCIA · Procesamiento del Lenguaje Natural 2025–26**

| Campo | Valor |
|---|---|
| Dominio | [Completar] |
| Tarea implementada | [T? — Nombre de la tarea] |
| Dataset | [Nombre y fuente] |
| Modelos comparados | Línea base trivial · TF-IDF + SVM · TF-IDF + LR |
| Entorno | Local CPU |
| Autores | [Nombre 1, Nombre 2, ...] |

---
**Cómo usar este notebook.**  
Ejecutar las celdas en orden. Cada sección corresponde a un epígrafe del informe de la Entrega 3.  
Los bloques marcados con `# TODO` requieren adaptación al dominio concreto.

## 0. Entorno y dependencias

In [1]:
# Instalar dependencias si no están disponibles
# Descomentar según necesidad

# !pip install scikit-learn pandas numpy matplotlib seaborn datasets
# !pip install nltk spacy
# !python -m spacy download es_core_news_sm   # modelo español
# !python -m nltk.downloader stopwords punkt

In [2]:
# ── Imports estándar ──────────────────────────────────────
import os
import time
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ── Scikit-learn ──────────────────────────────────────────
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.dummy import DummyClassifier
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)
from sklearn.utils import resample

# ── Reproducibilidad ──────────────────────────────────────
SEED = 42
np.random.seed(SEED)

print('Entorno listo.')

ModuleNotFoundError: No module named 'numpy'

## 1. Carga de datos

> **Nota.** Adaptar esta sección al dataset elegido.  
> Opciones habituales: HuggingFace `datasets`, CSV local, scraping propio.

In [ ]:
# ── Opción A: HuggingFace datasets ───────────────────────
# from datasets import load_dataset
# raw = load_dataset("[nombre-dataset]", "[config]")
# df_train = raw["train"].to_pandas()
# df_test  = raw["test"].to_pandas()

# ── Opción B: CSV local ───────────────────────────────────
# DATA_PATH = "data/dataset.csv"
# df = pd.read_csv(DATA_PATH)

# TODO: cargar el dataset real y asignar a df_train / df_test
# Columnas mínimas esperadas: 'text' (str) y 'label' (str o int)

# ── Placeholder para desarrollo ───────────────────────────
# Eliminar cuando se cargue el dataset real
df_train = pd.DataFrame({'text': [], 'label': []})
df_test  = pd.DataFrame({'text': [], 'label': []})

print(f"Train: {len(df_train):,} instancias")
print(f"Test:  {len(df_test):,} instancias")

## 2. Análisis exploratorio (EDA)

Objetivo: entender el corpus antes de modelar. Los hallazgos aquí deben conectar con la caracterización lingüística de la Entrega 1.

In [ ]:
# ── Distribución de clases ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (df, split) in zip(axes, [(df_train, 'Train'), (df_test, 'Test')]):
    if len(df) > 0:
        counts = df['label'].value_counts()
        ax.bar(counts.index.astype(str), counts.values, color='steelblue')
        ax.set_title(f'Distribución de clases — {split}')
        ax.set_xlabel('Clase')
        ax.set_ylabel('Frecuencia')
    else:
        ax.text(0.5, 0.5, 'Sin datos', ha='center', va='center')
        ax.set_title(f'Distribución de clases — {split}')

plt.tight_layout()
plt.savefig('fig_distribucion_clases.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Longitud de textos ────────────────────────────────────
if len(df_train) > 0:
    df_train['n_tokens'] = df_train['text'].str.split().str.len()

    fig, ax = plt.subplots(figsize=(8, 4))
    df_train['n_tokens'].hist(bins=50, ax=ax, color='steelblue', edgecolor='white')
    ax.set_title('Distribución de longitud de textos (tokens)')
    ax.set_xlabel('Número de tokens')
    ax.set_ylabel('Frecuencia')
    plt.tight_layout()
    plt.savefig('fig_longitud_textos.png', dpi=150, bbox_inches='tight')
    plt.show()

    print(df_train['n_tokens'].describe().round(1))

In [ ]:
# ── Ejemplos por clase ────────────────────────────────────
# TODO: ajustar N_EJEMPLOS y la columna 'label' al dominio
N_EJEMPLOS = 2

if len(df_train) > 0:
    for clase in df_train['label'].unique():
        print(f"\n{'─'*60}")
        print(f"Clase: {clase}")
        print('─'*60)
        muestra = df_train[df_train['label'] == clase].sample(
            min(N_EJEMPLOS, len(df_train[df_train['label'] == clase])),
            random_state=SEED
        )['text']
        for i, texto in enumerate(muestra, 1):
            print(f"  [{i}] {texto[:300]}")

## 3. Preprocesamiento

Documentar cada decisión de preprocesamiento y su justificación lingüística.

In [ ]:
import re
import string

# ── Stopwords ─────────────────────────────────────────────
# IMPORTANTE: preservar negaciones si son relevantes para la tarea
# (p.ej. 'no', 'nunca', 'jamás' en análisis de sentimiento)

# Opción A: NLTK
# import nltk
# from nltk.corpus import stopwords
# STOPWORDS = set(stopwords.words('spanish'))
# NEGACIONES = {'no', 'nunca', 'jamás', 'tampoco', 'ni'}
# STOPWORDS -= NEGACIONES   # preservar negaciones

# Opción B: lista propia
# STOPWORDS = {}

def limpiar_texto(texto: str) -> str:
    """
    Pipeline de preprocesamiento básico.
    Adaptar al dominio: añadir o quitar pasos según justificación lingüística.
    """
    # TODO: ajustar pasos al dominio
    texto = texto.lower()
    texto = re.sub(r'http\S+|www\S+', ' ', texto)        # eliminar URLs
    texto = re.sub(r'@\w+|#\w+', ' ', texto)              # menciones y hashtags
    texto = re.sub(r'[^\w\s]', ' ', texto)                # puntuación
    texto = re.sub(r'\d+', ' ', texto)                    # números
    texto = re.sub(r'\s+', ' ', texto).strip()            # espacios múltiples
    return texto

# Aplicar al corpus
if len(df_train) > 0:
    df_train['text_clean'] = df_train['text'].apply(limpiar_texto)
    df_test['text_clean']  = df_test['text'].apply(limpiar_texto)

    # Verificar un ejemplo
    idx = df_train.index[0]
    print("Original:", df_train.loc[idx, 'text'][:200])
    print("Limpio:  ", df_train.loc[idx, 'text_clean'][:200])
else:
    df_train['text_clean'] = df_train['text']
    df_test['text_clean']  = df_test['text']
    print("[Sin datos — preprocesamiento definido pero no aplicado]")

## 4. Modelos

Se comparan tres modelos bajo las mismas condiciones de evaluación:

| ID | Modelo | Rol |
|---|---|---|
| M0 | Clase mayoritaria | Línea base trivial (referencia mínima obligatoria) |
| M1 | TF-IDF + SVM lineal | Candidato A |
| M2 | TF-IDF + Regresión Logística | Candidato B |

In [ ]:
# ── Preparar arrays ───────────────────────────────────────
TEXT_COL = 'text_clean'   # o 'text' si no se aplica preprocesamiento
LABEL_COL = 'label'

if len(df_train) > 0:
    X_train = df_train[TEXT_COL].tolist()
    y_train = df_train[LABEL_COL].tolist()
    X_test  = df_test[TEXT_COL].tolist()
    y_test  = df_test[LABEL_COL].tolist()
else:
    # Placeholder: sustituir por datos reales
    X_train, y_train, X_test, y_test = [], [], [], []
    print("[Sin datos — definir X_train, y_train, X_test, y_test]")

In [ ]:
# ── Parámetros de TF-IDF ──────────────────────────────────
# Documentar cada decisión y su justificación
TFIDF_PARAMS = dict(
    max_features=50_000,
    ngram_range=(1, 2),     # unigramas + bigramas
    sublinear_tf=True,      # log(TF) reduce el peso de términos muy frecuentes
    min_df=2,               # ignorar términos que aparecen menos de 2 veces
    max_df=0.95,            # ignorar términos casi omnipresentes
    # TODO: añadir stop_words si se usa lista propia
)

# ── Definición de pipelines ───────────────────────────────
modelos = {
    'M0_baseline': Pipeline([
        ('clf', DummyClassifier(strategy='most_frequent', random_state=SEED))
    ]),
    'M1_tfidf_svm': Pipeline([
        ('tfidf', TfidfVectorizer(**TFIDF_PARAMS)),
        ('clf',   LinearSVC(C=1.0, max_iter=2000, random_state=SEED))
    ]),
    'M2_tfidf_lr': Pipeline([
        ('tfidf', TfidfVectorizer(**TFIDF_PARAMS)),
        ('clf',   LogisticRegression(C=1.0, max_iter=1000,
                                     solver='lbfgs', multi_class='auto',
                                     random_state=SEED))
    ]),
}

print(f"{len(modelos)} pipelines definidos.")

## 5. Entrenamiento y evaluación

In [ ]:
resultados = []

for nombre, pipeline in modelos.items():
    if len(X_train) == 0:
        print(f"[{nombre}] Sin datos — saltar.")
        continue

    t0 = time.time()
    pipeline.fit(X_train, y_train)
    t_train = time.time() - t0

    t0 = time.time()
    y_pred = pipeline.predict(X_test)
    t_inf  = time.time() - t0

    acc      = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average='macro', zero_division=0)
    f1_w     = f1_score(y_test, y_pred, average='weighted', zero_division=0)

    resultados.append({
        'modelo':    nombre,
        'accuracy':  round(acc, 4),
        'f1_macro':  round(f1_macro, 4),
        'f1_weighted': round(f1_w, 4),
        't_train_s': round(t_train, 2),
        't_inf_ms':  round(t_inf / len(X_test) * 1000, 2) if X_test else 0,
    })

    print(f"\n{'─'*50}")
    print(f"Modelo: {nombre}")
    print(f"  Accuracy : {acc:.4f}")
    print(f"  F1-macro : {f1_macro:.4f}")
    print(f"  F1-weighted: {f1_w:.4f}")
    print(f"  Entrenamiento: {t_train:.2f}s")
    if X_test:
        print(f"  Inferencia/instancia: {t_inf/len(X_test)*1000:.2f} ms")

    if len(X_test) > 0:
        print(f"\n{classification_report(y_test, y_pred, zero_division=0)}")

In [ ]:
# ── Tabla resumen ─────────────────────────────────────────
if resultados:
    df_res = pd.DataFrame(resultados).set_index('modelo')
    display(df_res)
    df_res.to_csv('resultados_comparativa.csv')
    print("\nGuardado: resultados_comparativa.csv")
else:
    print("[Sin resultados — cargar datos primero]")

In [ ]:
# ── Gráfico comparativo ───────────────────────────────────
if resultados:
    df_plot = pd.DataFrame(resultados)
    x = np.arange(len(df_plot))
    width = 0.28

    fig, ax = plt.subplots(figsize=(9, 5))
    ax.bar(x - width, df_plot['accuracy'],  width, label='Accuracy',   color='#2196F3')
    ax.bar(x,          df_plot['f1_macro'],  width, label='F1-macro',   color='#FF9800')
    ax.bar(x + width, df_plot['f1_weighted'],width, label='F1-weighted',color='#4CAF50')

    ax.set_xticks(x)
    ax.set_xticklabels(df_plot['modelo'], rotation=15, ha='right')
    ax.set_ylim(0, 1.05)
    ax.set_ylabel('Puntuación')
    ax.set_title('Comparativa de modelos')
    ax.legend()
    ax.grid(axis='y', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig('fig_comparativa_modelos.png', dpi=150, bbox_inches='tight')
    plt.show()

## 6. Matrices de confusión

In [ ]:
def plot_confusion(y_true, y_pred, titulo, ax):
    if len(y_true) == 0:
        ax.text(0.5, 0.5, 'Sin datos', ha='center', va='center')
        ax.set_title(titulo)
        return
    clases = sorted(set(y_true))
    cm = confusion_matrix(y_true, y_pred, labels=clases)
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    sns.heatmap(cm_norm, annot=cm, fmt='d', cmap='Blues',
                xticklabels=clases, yticklabels=clases, ax=ax,
                linewidths=0.5, vmin=0, vmax=1)
    ax.set_xlabel('Predicho')
    ax.set_ylabel('Real')
    ax.set_title(titulo)

if len(X_test) > 0:
    modelos_graficar = ['M1_tfidf_svm', 'M2_tfidf_lr']
    fig, axes = plt.subplots(1, len(modelos_graficar),
                             figsize=(6 * len(modelos_graficar), 5))
    if len(modelos_graficar) == 1:
        axes = [axes]

    for ax, nombre in zip(axes, modelos_graficar):
        if nombre in modelos:
            y_pred = modelos[nombre].predict(X_test)
            plot_confusion(y_test, y_pred, nombre, ax)

    plt.tight_layout()
    plt.savefig('fig_confusion_matrices.png', dpi=150, bbox_inches='tight')
    plt.show()
else:
    print("[Sin datos de test]")

## 7. Análisis de errores

Cada patrón de error debe conectar con el análisis lingüístico de la Entrega 1.  
El modelo de referencia para el análisis cualitativo es el mejor modelo cuantitativo.

In [ ]:
# ── Extraer errores del mejor modelo ─────────────────────
MODELO_ANALISIS = 'M1_tfidf_svm'   # TODO: cambiar al mejor modelo

if len(X_test) > 0 and MODELO_ANALISIS in modelos:
    y_pred_best = modelos[MODELO_ANALISIS].predict(X_test)

    df_analisis = df_test.copy()
    df_analisis['pred'] = y_pred_best
    df_analisis['error'] = df_analisis[LABEL_COL] != df_analisis['pred']

    errores = df_analisis[df_analisis['error']].copy()
    print(f"Total errores: {len(errores):,} / {len(df_test):,} "
          f"({len(errores)/len(df_test)*100:.1f}%)")
    print(f"\nDistribución de errores por clase real:")
    print(errores[LABEL_COL].value_counts())
else:
    errores = pd.DataFrame()
    print("[Sin datos de test o modelo no entrenado]")

In [ ]:
# ── Patrón 1: filtro para un tipo de error específico ────
# TODO: definir el criterio de filtrado según el dominio
# Ejemplos:
#   - reseñas con ironía: buscar patrones léxicos ('genial', 'increíble' + contexto negativo)
#   - textos cortos (< 10 tokens)
#   - clase real X predicha como Y

PATRON_1_DESCRIPCION = "[Describir el patrón — conectar con Entrega 1]"

if len(errores) > 0:
    # TODO: ajustar el filtro al patrón real
    patron1 = errores[errores['n_tokens'] < 10] if 'n_tokens' in errores.columns else errores

    print(f"Patrón 1: {PATRON_1_DESCRIPCION}")
    print(f"Instancias: {len(patron1)}")
    print()
    for _, row in patron1.head(5).iterrows():
        print(f"  Real: {row[LABEL_COL]} | Predicho: {row['pred']}")
        print(f"  Texto: {str(row['text'])[:200]}")
        print()

In [ ]:
# ── Patrón 2 ──────────────────────────────────────────────
PATRON_2_DESCRIPCION = "[Describir el patrón — conectar con Entrega 1]"

# TODO: definir filtro para patrón 2
if len(errores) > 0:
    patron2 = errores  # placeholder — reemplazar por filtro real
    print(f"Patrón 2: {PATRON_2_DESCRIPCION}")
    print(f"Instancias: {len(patron2)}")

In [ ]:
# ── Features más relevantes (interpretabilidad TF-IDF+SVM) ──
# Solo aplicable a LinearSVC y LogisticRegression

def top_features(pipeline, n=15):
    """Devuelve los n términos más discriminantes por clase."""
    vectorizer = pipeline.named_steps['tfidf']
    clf        = pipeline.named_steps['clf']
    vocab      = np.array(vectorizer.get_feature_names_out())

    if hasattr(clf, 'coef_'):
        coefs = clf.coef_
        clases = clf.classes_ if hasattr(clf, 'classes_') else range(coefs.shape[0])
        for i, clase in enumerate(clases):
            idx_top = np.argsort(coefs[i])[-n:][::-1]
            idx_bot = np.argsort(coefs[i])[:n]
            print(f"\nClase '{clase}' — términos más positivos:")
            print('  ', ', '.join(vocab[idx_top]))
            print(f"Clase '{clase}' — términos más negativos:")
            print('  ', ', '.join(vocab[idx_bot]))
    else:
        print("El clasificador no expone coef_ — no aplicable.")

if len(X_train) > 0 and 'M2_tfidf_lr' in modelos:
    print("=== Features más discriminantes — M2_tfidf_lr ===")
    top_features(modelos['M2_tfidf_lr'])

## 8. Desglose por subgrupo

Identificar degradaciones sistemáticas por idioma, categoría, longitud u otra variable relevante del dominio.

In [ ]:
# TODO: definir la columna de subgrupo relevante para el dominio
# Ejemplos: 'idioma', 'categoria', 'fuente'

SUBGRUPO_COL = None   # e.g., 'idioma'

if SUBGRUPO_COL and SUBGRUPO_COL in df_test.columns and len(X_test) > 0:
    df_analisis['pred'] = modelos['M1_tfidf_svm'].predict(X_test)
    for grupo, df_g in df_analisis.groupby(SUBGRUPO_COL):
        f1 = f1_score(df_g[LABEL_COL], df_g['pred'],
                      average='macro', zero_division=0)
        print(f"{SUBGRUPO_COL}={grupo:20s}  n={len(df_g):5,}  F1-macro={f1:.4f}")
else:
    print(f"[Definir SUBGRUPO_COL para activar este análisis]")

## 9. Resumen para el informe (Entrega 3)

Copiar los valores de esta celda directamente en las tablas del TDT.

In [ ]:
print("=" * 60)
print("RESUMEN DE RESULTADOS PARA EL TDT — ENTREGA 3")
print("=" * 60)

if resultados:
    df_res = pd.DataFrame(resultados)
    print(df_res.to_string(index=False))
    print()
    mejor = df_res.loc[df_res['f1_macro'].idxmax(), 'modelo']
    print(f"Mejor modelo (F1-macro): {mejor}")
    baseline_f1 = df_res.loc[df_res['modelo'] == 'M0_baseline', 'f1_macro'].values
    if len(baseline_f1) > 0:
        mejora = df_res.loc[df_res['f1_macro'].idxmax(), 'f1_macro'] - baseline_f1[0]
        print(f"Mejora sobre línea base: +{mejora:.4f}")
else:
    print("[Sin resultados — ejecutar sección 5 primero]")

print()
print("Figuras generadas:")
for f in [
    'fig_distribucion_clases.png',
    'fig_longitud_textos.png',
    'fig_comparativa_modelos.png',
    'fig_confusion_matrices.png',
]:
    existe = os.path.exists(f)
    print(f"  {'OK' if existe else '--'}  {f}")

---
## Notas de reproducibilidad

```
Python:      [completar — python --version]
scikit-learn:[completar — pip show scikit-learn]
pandas:      [completar]
numpy:       [completar]
Semilla:     42
Hardware:    Local CPU
```

Ver `requirements.txt` en la raíz del repositorio.

In [ ]:
# Generar requirements.txt desde el entorno actual
# !pip freeze > requirements.txt
# print("requirements.txt generado.")